In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import zipfile


zip_path = "/content/drive/MyDrive/cat vs dog.zip"


extract_path = "/content/cat_vs_dog"


with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Décompression terminée avec succès !")

Décompression terminée avec succès !


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

import matplotlib.pyplot as plt
import numpy as np

In [10]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(128, scale=(0.8, 1.0)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])
test_transform = transforms.Compose([
    transforms.Resize((128, 128)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])


In [ ]:
import os
from PIL import Image

dataset_path = "/content/cat_vs_dog/cat vs dog"

corrupted_images = []



for root, dirs, files in os.walk(dataset_path):

    for file in files:

        if file.lower().endswith((
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".webp"
        )):

            path = os.path.join(root, file)

            try:
                with Image.open(path) as img:
                    img.verify()

            except Exception:
                corrupted_images.append(path)




print("Images invalides trouvées :", len(corrupted_images))

for path in corrupted_images:
    print(path)




deleted = 0

for path in corrupted_images:

    if os.path.exists(path):

        os.remove(path)
        deleted += 1

print()
print("Images supprimées :", deleted)



remaining_invalid = []

for root, dirs, files in os.walk(dataset_path):

    for file in files:

        if file.lower().endswith((
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".webp"
        )):

            path = os.path.join(root, file)

            try:
                with Image.open(path) as img:
                    img.verify()

            except Exception:
                remaining_invalid.append(path)


print("Images invalides restantes :", len(remaining_invalid))

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Images invalides trouvées : 4
/content/cat_vs_dog/cat vs dog/PetImages/Dog/11702.jpg
/content/cat_vs_dog/cat vs dog/PetImages/Cat/666.jpg
/content/cat_vs_dog/cat vs dog/train/Dog/11702.jpg
/content/cat_vs_dog/cat vs dog/train/Cat/666.jpg

Images supprimées : 4
Images invalides restantes : 0


In [12]:
train_path = "/content/cat_vs_dog/cat vs dog/train"
test_path = "/content/cat_vs_dog/cat vs dog/test"

train_dataset = datasets.ImageFolder(
    train_path,
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    test_path,
    transform=test_transform
)

In [13]:
print("Classes:", train_dataset.classes)
print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

Classes: ['Cat', 'Dog']
Train: 19998
Test: 5000


In [14]:
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [15]:
class CNN(nn.Module):

    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc_layers = nn.Sequential(

            nn.Flatten(),

            nn.Linear(128 * 16 * 16, 128),
            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(128, 2)
        )

    def forward(self, x):

        x = self.conv_layers(x)
        x = self.fc_layers(x)

        return x

In [16]:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



model = CNN().to(device)

print(model)

CNN(
  (conv_layers): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=2, bias=True)
  )
)


In [17]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
num_epochs = 50

train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []

checkpoint_path = "/content/cat_vs_dog/cat_dog_checkpoint.pth"

for epoch in range(num_epochs):


    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

      
        optimizer.zero_grad()

       
        outputs = model(images)

       
        loss = criterion(outputs, labels)

     
        loss.backward()

       
        optimizer.step()

       
        running_loss += loss.item()

      
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total


    model.eval()

    test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            test_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = test_loss / len(test_loader)
    val_accuracy = 100 * correct / total


  
  
    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)


   
    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_accuracy:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.2f}%"
    )


 
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy
    }, checkpoint_path)




print("\nEntraînement terminé !")
print(f"Train Accuracy finale : {train_accuracies[-1]:.2f}%")
print(f"Val Accuracy finale   : {val_accuracies[-1]:.2f}%")

Epoch [1/50] | Train Loss: 0.6226 | Train Accuracy: 64.53% | Val Loss: 0.6200 | Val Accuracy: 67.46%
Epoch [2/50] | Train Loss: 0.5393 | Train Accuracy: 73.06% | Val Loss: 0.4897 | Val Accuracy: 76.34%
Epoch [3/50] | Train Loss: 0.4786 | Train Accuracy: 77.26% | Val Loss: 0.4417 | Val Accuracy: 79.36%
Epoch [4/50] | Train Loss: 0.4421 | Train Accuracy: 79.85% | Val Loss: 0.3998 | Val Accuracy: 81.90%
Epoch [5/50] | Train Loss: 0.4180 | Train Accuracy: 81.44% | Val Loss: 0.4602 | Val Accuracy: 78.92%
Epoch [6/50] | Train Loss: 0.3872 | Train Accuracy: 82.75% | Val Loss: 0.3735 | Val Accuracy: 83.22%
Epoch [7/50] | Train Loss: 0.3654 | Train Accuracy: 84.00% | Val Loss: 0.3454 | Val Accuracy: 84.80%
Epoch [8/50] | Train Loss: 0.3492 | Train Accuracy: 84.93% | Val Loss: 0.3836 | Val Accuracy: 83.28%
Epoch [9/50] | Train Loss: 0.3351 | Train Accuracy: 85.39% | Val Loss: 0.3633 | Val Accuracy: 84.48%
Epoch [10/50] | Train Loss: 0.3211 | Train Accuracy: 86.21% | Val Loss: 0.3532 | Val Accura

KeyboardInterrupt: 